# P1a — G1 gate + E1 profile

## 0. Parameters

In [1]:
import os
from pathlib import Path

# ── repo layout: walk up from CWD to find sft/ ────────────────────────────────
_p = Path().resolve()
REPO_ROOT = _p  # fallback
for _ in range(6):
    if (_p / "sft").is_dir():
        REPO_ROOT = _p
        break
    _p = _p.parent

SFT_DIR      = REPO_ROOT / "sft"
CSV_PATH     = REPO_ROOT / "data" / "processed" / "integrated_conversations_ner_validated.csv"
CONFIG_DIR   = SFT_DIR / "config"
REPORTS_DIR  = SFT_DIR / "reports"
WORK_E1_DIR  = SFT_DIR / "work" / "e1"
SYSTEM_PROMPT_PATH = CONFIG_DIR / "system_prompt_v1.txt"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_E1_DIR.mkdir(parents=True, exist_ok=True)

# ── limit (set via env var for fast dev runs) ──────────────────────────────────
_limit_env      = os.environ.get("P1A_LIMIT_DIALOGUES", "").strip()
LIMIT_DIALOGUES = int(_limit_env) if _limit_env else None   # None = full dataset

# ── G1 expected values (from ARCHITECTURE.md D1 + DATASET.md) ──────────────────
G1_EXPECTED = {
    "total_rows":          275_639,
    "total_dialogues":     123_256,
    "assistant_turns":     137_880,
    "user_turns":          137_759,
    "null_utterances":     198,
    "empty_entities_rows": 46_889,
    # per-source: (turns, dialogues)
    "source_MedDialog":           (224_328, 112_164),
    "source_HealthChat-LMSYS":    ( 28_183,   6_828),
    "source_HealthChat-WildChat": ( 23_128,   4_264),
    # structural checks
    "dialogue_id_to_source_unique": True,
    "pk_unique":                    True,
    "meddialog_exactly_2_turns":    True,
}

# ── E1 constants ───────────────────────────────────────────────────────────────
TOKEN_BINS = [0, 10, 20, 50, 100, 200, 500, 1000, 2000, 99999]

TEMPLATE_PATTERNS = [
    r"hope i have answered",
    r"chat doctor",
    r"chatdoctor",
    r"thanks for your query",
    r"i am dr",
    r"as your doctor",
    r"hi i am",
    r"hello i am",
    r"dear friend",
]

CREDENTIAL_PATTERN = r"(?:i am dr|as your doctor|i['\'']m a doctor|i am a doctor)"

print(f"REPO_ROOT      : {REPO_ROOT}")
print(f"CSV_PATH exists: {CSV_PATH.exists()}")
print(f"LIMIT_DIALOGUES: {LIMIT_DIALOGUES!r}")
print(f"G1_EXPECTED keys: {sorted(G1_EXPECTED.keys())}")


REPO_ROOT      : C:\Users\nirmi\Desktop\Capstone
CSV_PATH exists: True
LIMIT_DIALOGUES: None
G1_EXPECTED keys: ['assistant_turns', 'dialogue_id_to_source_unique', 'empty_entities_rows', 'meddialog_exactly_2_turns', 'null_utterances', 'pk_unique', 'source_HealthChat-LMSYS', 'source_HealthChat-WildChat', 'source_MedDialog', 'total_dialogues', 'total_rows', 'user_turns']


## 1. Common — constants, helpers, loaders

In [2]:
import gc
import hashlib
import json
import re
import time
import unicodedata
from typing import Optional

import pandas as pd

# ── timing registry ────────────────────────────────────────────────────────────
_SECTION_TIMES: dict = {}
_SECTION_START: dict = {}

def section_start(name: str) -> None:
    _SECTION_START[name] = time.perf_counter()

def section_end(name: str) -> float:
    elapsed = time.perf_counter() - _SECTION_START[name]
    _SECTION_TIMES[name] = elapsed
    return elapsed

# ── peak-memory helper (tracemalloc proxy) ─────────────────────────────────────
import tracemalloc
tracemalloc.start()

def peak_mem_mb() -> float:
    _, peak = tracemalloc.get_traced_memory()
    return peak / 1_048_576

# ── normalize_text  (R6) ───────────────────────────────────────────────────────
_BLANK_RUN = re.compile(r"\n{3,}")

def normalize_text(text: str) -> str:
    """R6: Unicode NFC, strip leading/trailing whitespace, collapse blank-line runs."""
    text = unicodedata.normalize("NFC", text)
    text = text.strip()
    text = _BLANK_RUN.sub("\n\n", text)
    return text

# ── load_turns — streaming chunked reader ─────────────────────────────────────
_DTYPES = {
    "dialogue_id":        "str",
    "turn_id":            "Int64",
    "speaker":            "str",
    "utterance":          "object",
    "source_dataset":     "str",
    "original_id":        "str",
    "extracted_entities": "object",
    "entity_labels":      "object",
    "entity_count":       "Int64",
}

def load_turns(
    csv_path: Path,
    usecols: Optional[list] = None,
    chunksize: int = 50_000,
    limit: Optional[int] = None,
) -> pd.DataFrame:
    """
    Read the validated CSV in chunks.
    limit: stop after this many unique dialogues (None = full dataset).
         Defaults to the module-level LIMIT_DIALOGUES.
    """
    effective_limit = limit if limit is not None else LIMIT_DIALOGUES
    cols  = usecols or list(_DTYPES.keys())
    dtype = {c: _DTYPES[c] for c in cols if c in _DTYPES}
    chunks: list = []
    seen_dialogues: set = set()

    reader = pd.read_csv(
        csv_path,
        usecols=cols,
        dtype=dtype,
        chunksize=chunksize,
        encoding="utf-8",
        low_memory=False,
    )
    for chunk in reader:
        if effective_limit is not None:
            # Add new dialogue IDs up to the limit only
            for _did in chunk["dialogue_id"].dropna().unique():
                if len(seen_dialogues) < effective_limit:
                    seen_dialogues.add(_did)
            # Keep only rows whose dialogue is in the accepted set
            chunk = chunk[chunk["dialogue_id"].isin(seen_dialogues)]
            if len(chunk):
                chunks.append(chunk)
            if len(seen_dialogues) >= effective_limit:
                break
        else:
            chunks.append(chunk)

    return pd.concat(chunks, ignore_index=True)

# ── report writer ─────────────────────────────────────────────────────────────
def write_json_report(path: Path, data: dict) -> None:
    """Write a JSON report with sorted keys and ensure_ascii=False."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        json.dump(data, fh, sort_keys=True, ensure_ascii=False, indent=2)
    print(f"  Wrote: {path}")

# ── D0 — load and hash system prompt ──────────────────────────────────────────
_sp_text = SYSTEM_PROMPT_PATH.read_text(encoding="utf-8")
SYSTEM_PROMPT_SHA256 = hashlib.sha256(_sp_text.encode("utf-8")).hexdigest()
print(f"SYSTEM_PROMPT_V1 sha256 : {SYSTEM_PROMPT_SHA256}")
print(f"SYSTEM_PROMPT_V1 length : {len(_sp_text)} chars")
del _sp_text

print("Section 1 helpers loaded.")


SYSTEM_PROMPT_V1 sha256 : e68af7244b7f6332620cd69a67cd4c17e800e5b59401067765cb1903ffa5b0a4
SYSTEM_PROMPT_V1 length : 508 chars
Section 1 helpers loaded.


## 2. Rules — cut_at_first_defect (R1–R3, R6)

In [3]:
import math
from typing import Any

def cut_at_first_defect(
    turns: list,
) -> tuple:
    """
    R1-R3 applied to a single dialogue's turns (pre-sorted by turn_id).
    Returns (valid_prefix, defect_reason_or_None).

    R2: Valid = turn_id contiguous from 0, roles alternate starting with
        'user', every utterance non-null and non-empty after R6 normalize.
    R3: Cut at first defect; keep longest prefix ending on assistant turn.
        Return empty list if no full exchange remains.

    Defect reasons:
      'non_contiguous_turn_id'
      'wrong_role_order'
      'null_or_empty_utterance'
    """
    if not turns:
        return [], "empty_dialogue"

    valid: list = []
    defect: Optional[str] = None

    for i, t in enumerate(turns):
        # R2a: contiguous turn_id
        if t["turn_id"] != i:
            defect = "non_contiguous_turn_id"
            break

        # R2b: alternation starting with user
        expected_role = "user" if i % 2 == 0 else "assistant"
        if t["speaker"] != expected_role:
            defect = "wrong_role_order"
            break

        # R2c: non-null, non-empty utterance after R6 normalize
        raw = t.get("utterance")
        if raw is None or (isinstance(raw, float) and math.isnan(raw)):
            defect = "null_or_empty_utterance"
            break
        normed = normalize_text(str(raw))
        if not normed:
            defect = "null_or_empty_utterance"
            break

        valid.append({**t, "utterance": normed})

    if defect is None:
        return valid, None

    # R3: trim to longest prefix ending on an assistant turn
    prefix = valid
    while prefix and prefix[-1]["speaker"] != "assistant":
        prefix = prefix[:-1]

    if not prefix:
        return [], defect
    return prefix, defect


def count_exchanges(turns: list) -> int:
    """Number of complete (user, assistant) pairs."""
    return sum(1 for t in turns if t["speaker"] == "assistant")


print("Section 2 rules defined: cut_at_first_defect, count_exchanges")


Section 2 rules defined: cut_at_first_defect, count_exchanges


## 3. Tests — in-notebook unit tests (synthetic fixtures only)

In [4]:
import io
import tempfile
import unittest


def _make_turns(*utterances, start_id=0):
    """Build a synthetic turn list with alternating user/assistant roles."""
    roles = ["user", "assistant"]
    return [
        {"turn_id": start_id + i, "speaker": roles[i % 2], "utterance": u}
        for i, u in enumerate(utterances)
    ]


# ─────────────────────────────────────────────────────────────────────────────
class TestNormalizeText(unittest.TestCase):

    def test_strips_whitespace(self):
        self.assertEqual(normalize_text("  hello  "), "hello")

    def test_nfc(self):
        nfd = "e\u0301"  # e + combining acute
        self.assertEqual(normalize_text(nfd), "\xe9")

    def test_collapses_blank_lines(self):
        self.assertEqual(normalize_text("a\n\n\n\nb"), "a\n\nb")

    def test_empty_string(self):
        self.assertEqual(normalize_text(""), "")

    def test_preserves_name_placeholder(self):
        self.assertIn("NAME_1", normalize_text("Hello NAME_1, how are you?"))


# ─────────────────────────────────────────────────────────────────────────────
class TestCutAtFirstDefect(unittest.TestCase):

    def test_perfect_dialogue_returned_intact(self):
        turns = _make_turns("hello", "hi there", "more text", "response here")
        result, defect = cut_at_first_defect(turns)
        self.assertIsNone(defect)
        self.assertEqual(len(result), 4)

    def test_empty_returns_empty(self):
        result, defect = cut_at_first_defect([])
        self.assertEqual(result, [])
        self.assertEqual(defect, "empty_dialogue")

    def test_null_utterance_no_prior_exchange(self):
        turns = _make_turns("hello", None, "extra", "response")
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "null_or_empty_utterance")
        self.assertEqual(result, [])

    def test_null_utterance_after_first_exchange(self):
        turns = _make_turns("Q1", "A1", None, "A2")
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "null_or_empty_utterance")
        self.assertEqual(len(result), 2)
        self.assertEqual(result[-1]["speaker"], "assistant")

    def test_non_contiguous_turn_id(self):
        turns = _make_turns("Q1", "A1")
        turns.append({"turn_id": 5, "speaker": "user", "utterance": "Q2"})
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "non_contiguous_turn_id")
        self.assertEqual(len(result), 2)

    def test_wrong_role_order_leading_assistant(self):
        turns = [
            {"turn_id": 0, "speaker": "assistant", "utterance": "hi"},
            {"turn_id": 1, "speaker": "user",      "utterance": "hello"},
        ]
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "wrong_role_order")
        self.assertEqual(result, [])

    def test_same_role_run(self):
        turns = _make_turns("Q1", "A1")
        turns.append({"turn_id": 2, "speaker": "assistant", "utterance": "extra A"})
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "wrong_role_order")
        self.assertEqual(len(result), 2)

    def test_blank_after_strip(self):
        turns = _make_turns("Q1", "   ", "Q2", "A2")
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "null_or_empty_utterance")
        self.assertEqual(result, [])

    def test_nan_utterance(self):
        turns = _make_turns("Q1", "A1")
        turns.append({"turn_id": 2, "speaker": "user",      "utterance": float("nan")})
        turns.append({"turn_id": 3, "speaker": "assistant", "utterance": "A2"})
        result, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "null_or_empty_utterance")
        self.assertEqual(len(result), 2)

    def test_r6_normalize_applied(self):
        turns = _make_turns("  Q1  ", "  A1  ")
        result, defect = cut_at_first_defect(turns)
        self.assertIsNone(defect)
        self.assertEqual(result[0]["utterance"], "Q1")
        self.assertEqual(result[1]["utterance"], "A1")

    def test_single_user_turn_no_exchange(self):
        turns = _make_turns("hello only")
        result, defect = cut_at_first_defect(turns)
        self.assertIsNone(defect)
        self.assertEqual(count_exchanges(result), 0)


# ─────────────────────────────────────────────────────────────────────────────
class TestCountExchanges(unittest.TestCase):

    def test_two_turn_dialogue(self):
        self.assertEqual(count_exchanges(_make_turns("Q", "A")), 1)

    def test_six_turn_dialogue(self):
        self.assertEqual(count_exchanges(_make_turns("Q1","A1","Q2","A2","Q3","A3")), 3)

    def test_empty(self):
        self.assertEqual(count_exchanges([]), 0)


# ─────────────────────────────────────────────────────────────────────────────
class TestLoadTurns(unittest.TestCase):

    def _write_csv(self, tmpdir, rows):
        import csv
        p = Path(tmpdir) / "test.csv"
        cols = ["dialogue_id","turn_id","speaker","utterance",
                "source_dataset","original_id",
                "extracted_entities","entity_labels","entity_count"]
        with open(p, "w", encoding="utf-8", newline="\n") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for r in rows:
                w.writerow(r)
        return p

    def _make_rows(self, n_dialogues=5, turns_each=2):
        rows = []
        roles = ["user", "assistant"]
        for d in range(n_dialogues):
            did = f"dia_{d:04d}"
            for t in range(turns_each):
                rows.append({
                    "dialogue_id": did,
                    "turn_id": t,
                    "speaker": roles[t % 2],
                    "utterance": f"utterance d{d} t{t}",
                    "source_dataset": "MedDialog",
                    "original_id": f"orig_{d}",
                    "extracted_entities": "",
                    "entity_labels": "",
                    "entity_count": 0,
                })
        return rows

    def test_full_load(self):
        with tempfile.TemporaryDirectory() as td:
            csv_p = self._write_csv(td, self._make_rows(10, 2))
            # pass limit=None explicitly — does not touch module global
            df = load_turns(csv_p, limit=None)
            self.assertEqual(len(df), 20)

    def test_limit_dialogues(self):
        with tempfile.TemporaryDirectory() as td:
            csv_p = self._write_csv(td, self._make_rows(10, 2))
            # pass limit=3 explicitly — does not touch module global
            df = load_turns(csv_p, limit=3)
            self.assertLessEqual(df["dialogue_id"].nunique(), 3)


# ─────────────────────────────────────────────────────────────────────────────
class TestWriteJsonReport(unittest.TestCase):

    def test_roundtrip(self):
        with tempfile.TemporaryDirectory() as td:
            p = Path(td) / "sub" / "report.json"
            write_json_report(p, {"key_b": 2, "key_a": 1})
            loaded = json.loads(p.read_text(encoding="utf-8"))
            self.assertEqual(loaded["key_a"], 1)
            # sorted keys: key_a before key_b
            raw = p.read_text(encoding="utf-8")
            self.assertLess(raw.index('"key_a"'), raw.index('"key_b"'))

    def test_ensure_ascii_false(self):
        with tempfile.TemporaryDirectory() as td:
            p = Path(td) / "r.json"
            write_json_report(p, {"val": "h\u00e9llo"})
            raw = p.read_bytes()
            self.assertIn("h\u00e9llo".encode("utf-8"), raw)


# ─────────────────────────────────────────────────────────────────────────────
# Run — must pass before any real data is read
# ─────────────────────────────────────────────────────────────────────────────
_suite = unittest.TestLoader().loadTestsFromTestCase(TestNormalizeText)
_suite.addTests(unittest.TestLoader().loadTestsFromTestCase(TestCutAtFirstDefect))
_suite.addTests(unittest.TestLoader().loadTestsFromTestCase(TestCountExchanges))
_suite.addTests(unittest.TestLoader().loadTestsFromTestCase(TestLoadTurns))
_suite.addTests(unittest.TestLoader().loadTestsFromTestCase(TestWriteJsonReport))

_buf = io.StringIO()
_runner = unittest.TextTestRunner(stream=_buf, verbosity=2)
_result = _runner.run(_suite)

print(_buf.getvalue())
print(f"Tests run  : {_result.testsRun}")
print(f"Failures   : {len(_result.failures)}")
print(f"Errors     : {len(_result.errors)}")

assert _result.wasSuccessful(), (
    f"IN-NOTEBOOK TESTS FAILED — {len(_result.failures)} failures, "
    f"{len(_result.errors)} errors. Fix before proceeding.\n"
    + _buf.getvalue()
)
print("\u2713 All in-notebook tests passed — proceeding to real data.")


  Wrote: C:\Users\nirmi\AppData\Local\Temp\tmpqdw9ptjg\r.json
  Wrote: C:\Users\nirmi\AppData\Local\Temp\tmpqrczx0zk\sub\report.json
test_collapses_blank_lines (__main__.TestNormalizeText.test_collapses_blank_lines) ... ok
test_empty_string (__main__.TestNormalizeText.test_empty_string) ... ok
test_nfc (__main__.TestNormalizeText.test_nfc) ... ok
test_preserves_name_placeholder (__main__.TestNormalizeText.test_preserves_name_placeholder) ... ok
test_strips_whitespace (__main__.TestNormalizeText.test_strips_whitespace) ... ok
test_blank_after_strip (__main__.TestCutAtFirstDefect.test_blank_after_strip) ... ok
test_empty_returns_empty (__main__.TestCutAtFirstDefect.test_empty_returns_empty) ... ok
test_nan_utterance (__main__.TestCutAtFirstDefect.test_nan_utterance) ... ok
test_non_contiguous_turn_id (__main__.TestCutAtFirstDefect.test_non_contiguous_turn_id) ... ok
test_null_utterance_after_first_exchange (__main__.TestCutAtFirstDefect.test_null_utterance_after_first_exchange) ... ok
te

## 4. G1 — source-CSV gate checks

In [5]:
section_start("G1")

# ── Load all columns (needed for G1 + E1 reuse) ────────────────────────────────
print("Loading CSV for G1 …")
_g1_cols = [
    "dialogue_id", "turn_id", "speaker",
    "utterance", "source_dataset", "original_id",
    "extracted_entities", "entity_labels", "entity_count",
]
df_g1 = load_turns(CSV_PATH, usecols=_g1_cols)
print(f"  Loaded {len(df_g1):,} rows, {df_g1['dialogue_id'].nunique():,} dialogues")


Loading CSV for G1 …


  Loaded 275,639 rows, 123,256 dialogues


In [6]:
# ── Compute all G1 metrics ─────────────────────────────────────────────────────
measured: dict = {}

measured["total_rows"]          = int(len(df_g1))
measured["total_dialogues"]     = int(df_g1["dialogue_id"].nunique())
measured["assistant_turns"]     = int((df_g1["speaker"] == "assistant").sum())
measured["user_turns"]          = int((df_g1["speaker"] == "user").sum())
measured["null_utterances"]     = int(df_g1["utterance"].isna().sum())
measured["empty_entities_rows"] = int((df_g1["entity_count"].fillna(0) == 0).sum())

for src in ["MedDialog", "HealthChat-LMSYS", "HealthChat-WildChat"]:
    mask = df_g1["source_dataset"] == src
    measured[f"source_{src}"] = (int(mask.sum()), int(df_g1.loc[mask, "dialogue_id"].nunique()))

_did_src = df_g1.groupby("dialogue_id")["source_dataset"].nunique()
measured["dialogue_id_to_source_unique"] = bool((_did_src == 1).all())

measured["pk_unique"] = bool(not df_g1.duplicated(subset=["dialogue_id","turn_id"]).any())

_md    = df_g1[df_g1["source_dataset"] == "MedDialog"]
_mdcnt = _md.groupby("dialogue_id")["turn_id"].count()
measured["meddialog_exactly_2_turns"] = bool((_mdcnt == 2).all())

# ── Compare measured vs expected ───────────────────────────────────────────────
g1_rows   = []
all_match = True

for key in sorted(G1_EXPECTED.keys()):
    exp  = G1_EXPECTED[key]
    meas = measured.get(key, "MISSING")
    status = "PASS" if meas == exp else "MISMATCH"
    if status == "MISMATCH":
        all_match = False
    exp_str  = str(exp) if isinstance(exp, (bool, tuple)) else f"{exp:,}" if isinstance(exp, int) else str(exp)
    meas_str = str(meas) if isinstance(meas, (bool, tuple)) else f"{meas:,}" if isinstance(meas, int) else str(meas)
    g1_rows.append({"id": key, "expected": exp_str, "measured": meas_str, "status": status})

# ── Print compact table ────────────────────────────────────────────────────────
print(f"{'id':<42} {'expected':>24} {'measured':>24} status")
print("-" * 96)
for r in g1_rows:
    print(f"{r['id']:<42} {r['expected']:>24} {r['measured']:>24} {r['status']}")

overall = "PASS" if all_match else "MISMATCH"
print(f"\nG1 overall: {overall}")

_g1_elapsed = section_end("G1")
print(f"G1 elapsed : {_g1_elapsed:.1f}s")

# ── Write g1_report.json ───────────────────────────────────────────────────────
g1_report = {
    "overall":          overall,
    "checks":           g1_rows,
    "rows_loaded":      measured["total_rows"],
    "dialogues_loaded": measured["total_dialogues"],
    "limit_dialogues":  LIMIT_DIALOGUES,
    "elapsed_s":        round(_g1_elapsed, 2),
}
write_json_report(REPORTS_DIR / "g1_report.json", g1_report)
print("\ndf_g1 kept in memory for E1 (avoids second CSV read).")


id                                                         expected                 measured status
------------------------------------------------------------------------------------------------
assistant_turns                                             137,880                  137,880 PASS
dialogue_id_to_source_unique                                   True                     True PASS
empty_entities_rows                                          46,889                   48,768 MISMATCH
meddialog_exactly_2_turns                                      True                     True PASS
null_utterances                                                 198                      139 MISMATCH
pk_unique                                                      True                     True PASS
source_HealthChat-LMSYS                               (28183, 6828)            (28183, 6828) PASS
source_HealthChat-WildChat                            (23128, 4264)            (23128, 4264) PASS
source_MedD

## 5. E1 — profiling (D0–D5)

In [7]:
section_start("E1-total")
e1_report: dict = {"limit_dialogues": LIMIT_DIALOGUES}


In [8]:
# ── D0 — System prompt hash ────────────────────────────────────────────────────
section_start("E1-D0")
e1_report["D0_system_prompt"] = {
    "path":       str(SYSTEM_PROMPT_PATH.relative_to(REPO_ROOT)),
    "sha256":     SYSTEM_PROMPT_SHA256,
    "char_count": int(SYSTEM_PROMPT_PATH.stat().st_size),
}
print(f"D0 sha256      : {SYSTEM_PROMPT_SHA256}")
print(f"   char_count  : {e1_report['D0_system_prompt']['char_count']}")
print(f"   D0 elapsed  : {section_end('E1-D0'):.2f}s")


D0 sha256      : e68af7244b7f6332620cd69a67cd4c17e800e5b59401067765cb1903ffa5b0a4
   char_count  : 508
   D0 elapsed  : 0.00s


In [9]:
# ── D1 — Turn-length histogram (word-count proxy) ─────────────────────────────
section_start("E1-D1")
_utt = df_g1["utterance"].dropna().astype(str)
_wc  = _utt.str.split().str.len()

_hist_counts = {}
for lo, hi in zip(TOKEN_BINS[:-1], TOKEN_BINS[1:]):
    label = f"{lo}-{hi-1}" if hi < 99999 else f"{lo}+"
    _hist_counts[label] = int(((_wc >= lo) & (_wc < hi)).sum())

e1_report["D1_turn_length_histogram_words"] = {
    "bins":           _hist_counts,
    "total_non_null": int(len(_wc)),
    "mean":           round(float(_wc.mean()), 2),
    "p50":            int(_wc.median()),
    "p90":            int(_wc.quantile(0.90)),
    "p95":            int(_wc.quantile(0.95)),
    "p99":            int(_wc.quantile(0.99)),
    "max":            int(_wc.max()),
}
print("D1 turn-length histogram (words):")
for label, cnt in _hist_counts.items():
    print(f"  [{label:>12}] : {cnt:>8,}")
print(f"  p90={e1_report['D1_turn_length_histogram_words']['p90']}  "
      f"p95={e1_report['D1_turn_length_histogram_words']['p95']}  "
      f"p99={e1_report['D1_turn_length_histogram_words']['p99']}  "
      f"max={e1_report['D1_turn_length_histogram_words']['max']}")

_d1_tsv = WORK_E1_DIR / "d1_length_hist.tsv"
with open(_d1_tsv, "w", encoding="utf-8", newline="\n") as _fh:
    _fh.write("bin\tcount\n")
    for _lbl, _cnt in _hist_counts.items():
        _fh.write(f"{_lbl}\t{_cnt}\n")
print(f"  Written: {_d1_tsv}")

del _utt, _wc
gc.collect()
print(f"  D1 elapsed: {section_end('E1-D1'):.2f}s")


D1 turn-length histogram (words):
  [         0-9] :   12,501
  [       10-19] :    8,694
  [       20-49] :   14,342
  [       50-99] :  139,261
  [     100-199] :   81,394
  [     200-499] :   18,407
  [     500-999] :      845
  [   1000-1999] :       40
  [       2000+] :       16
  p90=172  p95=228  p99=395  max=5970
  Written: C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d1_length_hist.tsv
  D1 elapsed: 14.89s


In [10]:
# ── D2 — Language distribution (langid, sampled) ─────────────────────────────
section_start("E1-D2")
import langid as _langid

_user_turns  = df_g1[(df_g1["speaker"] == "user") & df_g1["utterance"].notna()]
_total_user  = len(_user_turns)
_LANG_SAMPLE = 20_000

if len(_user_turns) > _LANG_SAMPLE:
    _user_sample = _user_turns["utterance"].sample(n=_LANG_SAMPLE, random_state=42).astype(str)
    _sampled_n   = _LANG_SAMPLE
else:
    _user_sample = _user_turns["utterance"].astype(str)
    _sampled_n   = len(_user_sample)

_lang_counts: dict = {}
for _utt in _user_sample:
    _lang, _ = _langid.classify(_utt[:300])
    _lang_counts[_lang] = _lang_counts.get(_lang, 0) + 1

_lang_sorted = dict(sorted(_lang_counts.items(), key=lambda x: -x[1]))
_top10       = dict(list(_lang_sorted.items())[:10])
_en_count    = _lang_counts.get("en", 0)
_en_pct      = round(100 * _en_count / _sampled_n, 2) if _sampled_n else 0

e1_report["D2_language_distribution"] = {
    "sampled_user_turns": _sampled_n,
    "total_user_turns":   _total_user,
    "top10_langs":        _top10,
    "english_count":      _en_count,
    "english_pct":        _en_pct,
}
print(f"D2 language distribution (sampled {_sampled_n:,} / {_total_user:,} user turns):")
for _lang, _cnt in list(_top10.items())[:10]:
    print(f"  {_lang}: {_cnt:>7,}  ({100*_cnt/_sampled_n:.1f}%)")
print(f"  English: {_en_count:,} ({_en_pct}%)")

_d2_tsv = WORK_E1_DIR / "d2_language_dist.tsv"
with open(_d2_tsv, "w", encoding="utf-8", newline="\n") as _fh:
    _fh.write("lang\tcount\n")
    for _lang, _cnt in _lang_sorted.items():
        _fh.write(f"{_lang}\t{_cnt}\n")
print(f"  Written: {_d2_tsv}")

del _user_turns, _user_sample, _lang_counts, _lang_sorted
gc.collect()
print(f"  D2 elapsed: {section_end('E1-D2'):.2f}s")


D2 language distribution (sampled 20,000 / 137,717 user turns):
  en:  19,830  (99.2%)
  no:      32  (0.2%)
  de:      21  (0.1%)
  es:      18  (0.1%)
  fr:      12  (0.1%)
  nl:      12  (0.1%)
  it:      11  (0.1%)
  da:      10  (0.1%)
  mt:       9  (0.0%)
  pl:       5  (0.0%)
  English: 19,830 (99.15%)
  Written: C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d2_language_dist.tsv
  D2 elapsed: 57.65s


In [11]:
# ── D3 — Template pattern frequency ───────────────────────────────────────────
section_start("E1-D3")

_asst_turns       = df_g1[(df_g1["speaker"] == "assistant") & df_g1["utterance"].notna()]
_total_asst       = len(_asst_turns)
_asst_utt_lower   = _asst_turns["utterance"].astype(str).str.lower()

_template_counts: dict = {}
for _pat in TEMPLATE_PATTERNS:
    _template_counts[_pat] = int(_asst_utt_lower.str.contains(_pat, regex=True, na=False).sum())

_any_match = int(_asst_utt_lower.str.contains("|".join(TEMPLATE_PATTERNS), regex=True, na=False).sum())
_any_pct   = round(100 * _any_match / _total_asst, 4) if _total_asst else 0

e1_report["D3_template_patterns"] = {
    "total_assistant_turns": _total_asst,
    "pattern_counts":        _template_counts,
    "any_match_count":       _any_match,
    "any_match_pct":         _any_pct,
}
print(f"D3 template patterns ({_total_asst:,} assistant turns):")
for _pat, _cnt in _template_counts.items():
    print(f"  {_pat!r:<35}: {_cnt:>7,}  ({100*_cnt/_total_asst:.3f}%)")
print(f"  Any template match: {_any_match:,} ({_any_pct}%)")

_d3_tsv = WORK_E1_DIR / "d3_template_counts.tsv"
with open(_d3_tsv, "w", encoding="utf-8", newline="\n") as _fh:
    _fh.write("pattern\tcount\n")
    for _pat, _cnt in _template_counts.items():
        _fh.write(f"{_pat}\t{_cnt}\n")
print(f"  Written: {_d3_tsv}")

del _asst_turns, _asst_utt_lower
gc.collect()
print(f"  D3 elapsed: {section_end('E1-D3'):.2f}s")


D3 template patterns (137,783 assistant turns):
  'hope i have answered'             :  14,270  (10.357%)
  'chat doctor'                      :  69,767  (50.635%)
  'chatdoctor'                       :   6,116  (4.439%)
  'thanks for your query'            :   5,477  (3.975%)
  'i am dr'                          :       0  (0.000%)
  'as your doctor'                   :     232  (0.168%)
  'hi i am'                          :     181  (0.131%)
  'hello i am'                       :       6  (0.004%)
  'dear friend'                      :     785  (0.570%)
  Any template match: 81,090 (58.8534%)
  Written: C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d3_template_counts.tsv
  D3 elapsed: 2.66s


In [12]:
# ── D4 — Dialogue-length histogram (exchanges per dialogue) ────────────────────
section_start("E1-D4")

_dlg_turns  = df_g1.groupby("dialogue_id")["speaker"].apply(
    lambda s: int((s == "assistant").sum())
).rename("n_exchanges")

_exch_bins = {
    "1_exchange":  int((_dlg_turns == 1).sum()),
    "2_exchanges": int((_dlg_turns == 2).sum()),
    "3_to_5":      int(((_dlg_turns >= 3) & (_dlg_turns <= 5)).sum()),
    "6_to_10":     int(((_dlg_turns >= 6) & (_dlg_turns <= 10)).sum()),
    "11_plus":     int((_dlg_turns >= 11).sum()),
}
_multi_exch = int((_dlg_turns >= 2).sum())

e1_report["D4_dialogue_length_histogram"] = {
    "total_dialogues":        int(len(_dlg_turns)),
    "exchange_bins":          _exch_bins,
    "multi_exchange_dialogues": _multi_exch,
    "mean_exchanges":         round(float(_dlg_turns.mean()), 3),
    "p50_exchanges":          int(_dlg_turns.median()),
    "max_exchanges":          int(_dlg_turns.max()),
}
print(f"D4 dialogue-length histogram ({len(_dlg_turns):,} dialogues):")
for _lbl, _cnt in _exch_bins.items():
    print(f"  {_lbl:<20}: {_cnt:>8,}")
print(f"  Multi-exchange (>=2)  : {_multi_exch:,}")
print(f"  Mean exchanges        : {e1_report['D4_dialogue_length_histogram']['mean_exchanges']}")
print(f"  Max exchanges         : {e1_report['D4_dialogue_length_histogram']['max_exchanges']}")

# full distribution TSV
_dlg_hist_full = _dlg_turns.value_counts().sort_index()
_d4_tsv = WORK_E1_DIR / "d4_dialogue_length.tsv"
with open(_d4_tsv, "w", encoding="utf-8", newline="\n") as _fh:
    _fh.write("n_exchanges\tcount\n")
    for _n, _cnt in _dlg_hist_full.items():
        _fh.write(f"{_n}\t{_cnt}\n")
print(f"  Written: {_d4_tsv}")

del _dlg_turns, _dlg_hist_full
gc.collect()
print(f"  D4 elapsed: {section_end('E1-D4'):.2f}s")


D4 dialogue-length histogram (123,256 dialogues):
  1_exchange          :  118,384
  2_exchanges         :    1,962
  3_to_5              :    2,018
  6_to_10             :      698
  11_plus             :      194
  Multi-exchange (>=2)  : 4,872
  Mean exchanges        : 1.119
  Max exchanges         : 50
  Written: C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d4_dialogue_length.tsv
  D4 elapsed: 23.88s


In [13]:
# ── D5 — R3 cut-at-first-defect audit ─────────────────────────────────────────
section_start("E1-D5")

_defect_counts: dict = {
    "ok":                      0,
    "empty_dialogue":          0,
    "non_contiguous_turn_id":  0,
    "wrong_role_order":        0,
    "null_or_empty_utterance": 0,
}
_dialogues_truncated  = 0
_dialogues_dropped    = 0
_dialogues_ok         = 0
_exch_before          = 0
_exch_after           = 0

import pandas as _pd

_d5_df = df_g1[["dialogue_id","turn_id","speaker","utterance"]].copy()

for _did, _grp in _d5_df.groupby("dialogue_id", sort=False):
    _grp_s = _grp.sort_values("turn_id")
    _turns_in = []
    for _, _row in _grp_s.iterrows():
        _tid = int(_row["turn_id"]) if not _pd.isna(_row["turn_id"]) else -1
        _turns_in.append({
            "turn_id":   _tid,
            "speaker":   str(_row["speaker"]),
            "utterance": _row["utterance"],
        })
    _eb = count_exchanges(_turns_in)
    _exch_before += _eb

    _valid, _defect = cut_at_first_defect(_turns_in)
    _ea = count_exchanges(_valid)
    _exch_after += _ea

    if _defect is None:
        _dialogues_ok += 1
        _defect_counts["ok"] += 1
    else:
        _defect_counts[_defect] = _defect_counts.get(_defect, 0) + 1
        if _ea > 0:
            _dialogues_truncated += 1
        else:
            _dialogues_dropped += 1

_total_dias_d5 = _dialogues_ok + _dialogues_truncated + _dialogues_dropped

e1_report["D5_r3_defect_audit"] = {
    "total_dialogues_audited":    _total_dias_d5,
    "dialogues_ok":               _dialogues_ok,
    "dialogues_truncated":        _dialogues_truncated,
    "dialogues_dropped":          _dialogues_dropped,
    "defect_counts":              _defect_counts,
    "total_exchanges_before_cut": _exch_before,
    "total_exchanges_after_cut":  _exch_after,
    "exchange_retention_pct":     round(100 * _exch_after / _exch_before, 3) if _exch_before else 0,
}
print(f"D5 R3 defect audit ({_total_dias_d5:,} dialogues):")
print(f"  ok          : {_dialogues_ok:>8,}")
print(f"  truncated   : {_dialogues_truncated:>8,}")
print(f"  dropped     : {_dialogues_dropped:>8,}")
print("  Defect breakdown:")
for _reason, _cnt in _defect_counts.items():
    print(f"    {_reason:<35}: {_cnt:>7,}")
print(f"  Exchanges before cut : {_exch_before:,}")
print(f"  Exchanges after cut  : {_exch_after:,}")
print(f"  Exchange retention   : {e1_report['D5_r3_defect_audit']['exchange_retention_pct']}%")

_d5_tsv = WORK_E1_DIR / "d5_defect_audit.tsv"
with open(_d5_tsv, "w", encoding="utf-8", newline="\n") as _fh:
    _fh.write("reason\tcount\n")
    for _reason, _cnt in _defect_counts.items():
        _fh.write(f"{_reason}\t{_cnt}\n")
print(f"  Written: {_d5_tsv}")

del _d5_df
gc.collect()
print(f"  D5 elapsed: {section_end('E1-D5'):.2f}s")

_e1_total_t = section_end("E1-total")
print(f"E1 total elapsed: {_e1_total_t:.1f}s")


D5 R3 defect audit (123,256 dialogues):
  ok          :  122,918
  truncated   :      325
  dropped     :       13
  Defect breakdown:
    ok                                 : 122,918
    empty_dialogue                     :       0
    non_contiguous_turn_id             :     218
    wrong_role_order                   :       0
    null_or_empty_utterance            :     120
  Exchanges before cut : 137,880
  Exchanges after cut  : 136,755
  Exchange retention   : 99.184%
  Written: C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d5_defect_audit.tsv
  D5 elapsed: 98.84s
E1 total elapsed: 198.0s


In [14]:
# ── Write e1_profile.json ──────────────────────────────────────────────────────
e1_report["elapsed_s"] = round(_e1_total_t, 2)
write_json_report(REPORTS_DIR / "e1_profile.json", e1_report)

del df_g1
gc.collect()
print("df_g1 freed.")


  Wrote: C:\Users\nirmi\Desktop\Capstone\sft\reports\e1_profile.json
df_g1 freed.


## 6. Final status

In [15]:
import sys

_peak_mb = peak_mem_mb()
tracemalloc.stop()

print("=" * 70)
print("FINAL STATUS")
print("=" * 70)
print(f"G1 overall        : {overall}")
print(f"Tests run         : {_result.testsRun}  "
      f"failures={len(_result.failures)}  errors={len(_result.errors)}")
print()
print("Section runtimes:")
for _sec, _t in sorted(_SECTION_TIMES.items()):
    print(f"  {_sec:<22}: {_t:>7.1f}s")
print()
print(f"Peak traced memory: {_peak_mb:.1f} MB")
print(f"Python            : {sys.version}")
print()
print("Report files:")
for _p in sorted(REPORTS_DIR.glob("*.json")):
    print(f"  {_p}")
print()
print("E1 TSVs (sft/work/e1/):")
for _p in sorted(WORK_E1_DIR.glob("*.tsv")):
    print(f"  {_p}")
print()
assert overall in ("PASS", "MISMATCH"), f"Unexpected G1 overall: {overall!r}"
if overall == "MISMATCH":
    print("WARNING: G1 MISMATCH — review g1_report.json for details.")
else:
    print("\u2713 G1 PASS")
print("\u2713 Notebook complete.")


FINAL STATUS
G1 overall        : MISMATCH
Tests run         : 23  failures=0  errors=0

Section runtimes:
  E1-D0                 :     0.0s
  E1-D1                 :    14.9s
  E1-D2                 :    57.7s
  E1-D3                 :     2.7s
  E1-D4                 :    23.9s
  E1-D5                 :    98.8s
  E1-total              :   198.0s
  G1                    :     1.6s

Peak traced memory: 1644.1 MB
Python            : 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]

Report files:
  C:\Users\nirmi\Desktop\Capstone\sft\reports\e1_profile.json
  C:\Users\nirmi\Desktop\Capstone\sft\reports\g1_report.json

E1 TSVs (sft/work/e1/):
  C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d1_length_hist.tsv
  C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d2_language_dist.tsv
  C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d3_template_counts.tsv
  C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d4_dialogue_length.tsv
  C:\Users\nirmi\Desktop\Capstone\sft\work\e1\d5_